# 310. Huggin Face Tokenizers

- transformers에서는 tokenizer 학습 기능을 제공 않고, 모든 tokenizer는 pre-trained tokenizer라고 가정하므로 별도로 Hugging Face의 tokenizers를 이용하여 tokenizer를 학습해야 한다.

In [8]:
import tokenizers
tokenizers.__version__

'0.10.2'

In [10]:
print(dir(tokenizers))

['AddedToken', 'BertWordPieceTokenizer', 'ByteLevelBPETokenizer', 'CharBPETokenizer', 'EncodeInput', 'Encoding', 'Enum', 'InputSequence', 'List', 'NormalizedString', 'OffsetReferential', 'OffsetType', 'Offsets', 'PreTokenizedEncodeInput', 'PreTokenizedInputSequence', 'PreTokenizedString', 'Regex', 'SentencePieceBPETokenizer', 'SentencePieceUnigramTokenizer', 'SplitDelimiterBehavior', 'TextEncodeInput', 'TextInputSequence', 'Token', 'Tokenizer', 'Tuple', 'Union', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'decoders', 'implementations', 'models', 'normalizers', 'pre_tokenizers', 'processors', 'tokenizers', 'trainers']


In [9]:
print(dir(tokenizers.BertWordPieceTokenizer))

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'add_special_tokens', 'add_tokens', 'decode', 'decode_batch', 'decoder', 'enable_padding', 'enable_truncation', 'encode', 'encode_batch', 'from_file', 'get_vocab', 'get_vocab_size', 'id_to_token', 'model', 'no_padding', 'no_truncation', 'normalize', 'normalizer', 'num_special_tokens_to_add', 'padding', 'post_process', 'post_processor', 'pre_tokenizer', 'save', 'save_model', 'to_str', 'token_to_id', 'train', 'train_from_iterator', 'truncation']


## Bert WordPiece Tokenizer

In [95]:
from tokenizers import (ByteLevelBPETokenizer,
                        CharBPETokenizer,
                        SentencePieceBPETokenizer,
                        BertWordPieceTokenizer)

small_corpus = './data/very_small_alphabet_corpus.txt'

In [96]:
f = open('./data/very_small_alphabet_corpus.txt', 'r')
f.read()

'ABCDE ABC AC ABD\nDE AB ABC AF'

Huggingface의 Tokenizer trainer는 학습시 다음과 같은 인자를 갖는다

min_frequency : merge를 수행할 최소 빈도수, 5로 설정 시 5회 이상 등장한 pair만 수행한다  
vocab_size: 만들고자 하는 vocab의 size, 보통 '32000' 정도가 좋다고 알려져 있다  
show_progress : 학습 진행과정 show  
special_tokens : Tokenizer에 추가하고 싶은 special token 지정  
limit_alphabet : merge 수행 전 initial tokens이 유지되는 숫자 제한  
initial_alphabet : 꼭 포함됐으면 하는 initial alphabet, 이곳에 설정한 token은 학습되지 않고 그대로 포함되도록 설정된다.

In [97]:
from tokenizers import BertWordPieceTokenizer

bert_wordpiece_tokenizer = BertWordPieceTokenizer()
bert_wordpiece_tokenizer.train(
    files = [small_corpus],
    vocab_size = 10,
    min_frequency = 1,
    limit_alphabet = 1000,
    initial_alphabet = [],
    special_tokens = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"],
    show_progress = True,
    wordpieces_prefix = "##",
)

vocab = bert_wordpiece_tokenizer.get_vocab()

print(vocab)
print()
print(sorted(vocab, key=lambda x: vocab[x]))

{'b': 6, 'c': 7, '[SEP]': 3, 'a': 5, '##b': 11, '##c': 12, '##e': 15, 'd': 8, '##f': 13, '[PAD]': 0, '[CLS]': 2, 'f': 10, '##d': 14, '[MASK]': 4, 'e': 9, '[UNK]': 1}

['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]', 'a', 'b', 'c', 'd', 'e', 'f', '##b', '##c', '##f', '##d', '##e']


- encode() method를 이용하여 sentence를 tokenize

In [98]:
encoding = bert_wordpiece_tokenizer.encode('ABCDE')
print(encoding.tokens)
print(encoding.ids)

['a', '##b', '##c', '##d', '##e']
[5, 11, 12, 14, 15]


### 대문자 그대로 tokenize

- lowercase = False

In [99]:
bert_wordpiece_tokenizer = BertWordPieceTokenizer(
    lowercase = False)
bert_wordpiece_tokenizer.train(files=[small_corpus], vocab_size=10)
encoding = bert_wordpiece_tokenizer.encode('ABCDE')
print(encoding.tokens)
print(encoding.ids)

['A', '##B', '##C', '##D', '##E']
[5, 12, 13, 14, 15]


### vocab size 를 키우면 더 많은 subword 가 vocab으로 학습

In [100]:
bert_wordpiece_tokenizer = BertWordPieceTokenizer()
bert_wordpiece_tokenizer.train(
    files = [small_corpus],
    vocab_size = 20,
    min_frequency = 1,
    initial_alphabet = ['g'],
)
vocab = bert_wordpiece_tokenizer.get_vocab()
print(sorted(vocab, key=lambda x: vocab[x]))

['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]', 'a', 'b', 'c', 'd', 'e', 'f', 'g', '##b', '##d', '##f', '##c', '##e', 'ab', 'abc', 'af']


### sentence 를 list 로 입력 받아 batch encoding

In [101]:
encodings = bert_wordpiece_tokenizer.encode_batch(['ABCDE', 'abcd'])
print(encodings[0].tokens)
print(encodings[1].tokens)

['abc', '##d', '##e']
['abc', '##d']


In [102]:
bert_wordpiece_tokenizer.save_model(
    directory = './huggingface_output',
    prefix='very_small_bertwordpiece'
)

['./huggingface_output/very_small_bertwordpiece-vocab.txt']

In [103]:
bert_wordpiece_tokenizer.encode('ABCDE').tokens

['abc', '##d', '##e']

In [104]:
bert_wordpiece_tokenizer = BertWordPieceTokenizer(
    vocab = './huggingface_output/very_small_bertwordpiece-vocab.txt'
)

bert_wordpiece_tokenizer.encode('ABCDE').tokens

['[CLS]', 'abc', '##d', '##e', '[SEP]']

In [105]:
bert_wordpiece_tokenizer.encode('ABCDE', add_special_tokens=False).tokens

['abc', '##d', '##e']

### BertWordPieceTokenizer 는 두개의 문장을 pair 하는 기능 제공

In [106]:
bert_wordpiece_tokenizer.encode(
    sequence='abcde',
    pair = 'abcd'
).tokens

['[CLS]', 'abc', '##d', '##e', '[SEP]', 'abc', '##d', '[SEP]']

# 각 tokenizer 의 차이점 파악

## SentencePiece BPE Tokenizer

add_prefix_space=True 이면 문장의 맨 앞 단어에도 공백을 부여, False 이면 공백없이 시작하는 단어에는 ▁ 를 붙이지 않습니다.

In [107]:
sentencepiece_tokenizer = SentencePieceBPETokenizer(
    add_prefix_space = True)

sentencepiece_tokenizer.train(
    files = [small_corpus],
    vocab_size = 20,
    min_frequency = 1,
    special_tokens = ['<unk>'],)
vocab = sentencepiece_tokenizer.get_vocab()
print(sorted(vocab, key=lambda x: vocab[x]))

['<unk>', '\n', 'A', 'B', 'C', 'D', 'E', 'F', '▁', '▁A', '▁AB', '▁ABC', 'DE', 'D\n', '▁DE', '▁AC', '▁AF', '▁ABD\n', '▁ABCDE']


In [108]:
sentencepiece_tokenizer = SentencePieceBPETokenizer(
    add_prefix_space = False)

sentencepiece_tokenizer.train(
    files = [small_corpus],
    vocab_size = 20,
    min_frequency = 1,
    special_tokens = ['<unk>'],)
vocab = sentencepiece_tokenizer.get_vocab()
print(sorted(vocab, key=lambda x: vocab[x]))

['<unk>', '\n', 'A', 'B', 'C', 'D', 'E', 'F', '▁', '▁A', '▁AB', 'DE', '▁ABC', 'AB', 'CDE', 'D\n', '▁AC', '▁AF', '▁ABD\n', 'ABCDE']


## Character BPE Tokenizer

CharBPETokenizer 의 기본값은 공백/구두점으로 pre-tokenization 을 수행합니다.

In [109]:
charbpe_tokenizer = CharBPETokenizer(suffix='</w>')
charbpe_tokenizer.train(
    files = [small_corpus],
    vocab_size = 15,
    min_frequency = 1
)
charbpe_tokenizer.encode('ABCDE.ABC').tokens

['AB', 'C', 'DE</w>', 'ABC</w>']

In [113]:
charbpe_tokenizer = CharBPETokenizer(
    suffix='</w>',
    split_on_whitespace_only = True
)
charbpe_tokenizer.train(
    files = [small_corpus],
    vocab_size = 15,
    min_frequency = 1
)
charbpe_tokenizer.encode('ABCDE.ABC').tokens

['AB', 'C', 'D', 'E', 'ABC</w>']

In [114]:
charbpe_tokenizer.encode('ABCDEFGH').tokens

['AB', 'C', 'D', 'E', 'F']

## Byte-level BPE Tokenizer

- OpenAI GPT2 tokenizer  

- Byte-level BPE 는 글자가 아닌 byte 기준으로 BPE 를 적용하기 때문에 1 byte 로 표현되 는 글자 (알파벳, 숫자, 기호) 만 형태가 보존됩니다.

In [115]:
bytebpe_tokenizer = ByteLevelBPETokenizer(add_prefix_space=True)

bytebpe_tokenizer.train(files = [small_corpus],
    vocab_size = 1000, min_frequency = 1)

vocab = bytebpe_tokenizer.get_vocab()
print(sorted(vocab, key=lambda x: vocab[x]))

['!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '<', '=', '>', '?', '@', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', '\\', ']', '^', '_', '`', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '{', '|', '}', '~', '¡', '¢', '£', '¤', '¥', '¦', '§', '¨', '©', 'ª', '«', '¬', '®', '¯', '°', '±', '²', '³', '´', 'µ', '¶', '·', '¸', '¹', 'º', '»', '¼', '½', '¾', '¿', 'À', 'Á', 'Â', 'Ã', 'Ä', 'Å', 'Æ', 'Ç', 'È', 'É', 'Ê', 'Ë', 'Ì', 'Í', 'Î', 'Ï', 'Ð', 'Ñ', 'Ò', 'Ó', 'Ô', 'Õ', 'Ö', '×', 'Ø', 'Ù', 'Ú', 'Û', 'Ü', 'Ý', 'Þ', 'ß', 'à', 'á', 'â', 'ã', 'ä', 'å', 'æ', 'ç', 'è', 'é', 'ê', 'ë', 'ì', 'í', 'î', 'ï', 'ð', 'ñ', 'ò', 'ó', 'ô', 'õ', 'ö', '÷', 'ø', 'ù', 'ú', 'û', 'ü', 'ý', 'þ', 'ÿ', 'Ā', 'ā', 'Ă', 'ă', 'Ą', 'ą', 'Ć', 'ć', 'Ĉ', 'ĉ', 'Ċ', 'ċ'

- 띄어쓰기로 시작하는 단어 앞에 Ġ 를 prefix 로 부착합니다.

In [116]:
bytebpe_tokenizer.encode('ABCDE ABC').tokens

['ĠABCDE', 'ĠABC']

## 코로나19 관련 뉴스를 학습

In [118]:
from tokenizers import (ByteLevelBPETokenizer,
                        CharBPETokenizer,
                        SentencePieceBPETokenizer,
                        BertWordPieceTokenizer)

corpus_path = './data/2020-07-29_covid_news_sents.txt'
vocab_size = 3000

In [122]:
byte_level_bpe_tokenizer = ByteLevelBPETokenizer()
byte_level_bpe_tokenizer.train(files=[corpus_path], vocab_size=vocab_size)
byte_level_bpe_tokenizer.save_model(directory='./huggingface_output', prefix='ByteLevelBPETokenizer-covid')

['./huggingface_output/ByteLevelBPETokenizer-covid-vocab.json',
 './huggingface_output/ByteLevelBPETokenizer-covid-merges.txt']

In [123]:
char_bpe_tokenizer = CharBPETokenizer()
char_bpe_tokenizer.train(files=[corpus_path], vocab_size=vocab_size)
char_bpe_tokenizer.save_model(directory='./huggingface_output', prefix='CharBPETokenizer-covid')

['./huggingface_output/CharBPETokenizer-covid-vocab.json',
 './huggingface_output/CharBPETokenizer-covid-merges.txt']

In [126]:
sentencepiece_bpe_tokenizer = SentencePieceBPETokenizer()
sentencepiece_bpe_tokenizer.train(files=[corpus_path], vocab_size=vocab_size)
sentencepiece_bpe_tokenizer.save_model(directory='./huggingface_output', prefix='SentencePieceBPETokenizer-covid')

['./huggingface_output/SentencePieceBPETokenizer-covid-vocab.json',
 './huggingface_output/SentencePieceBPETokenizer-covid-merges.txt']

In [127]:
bert_wordpiece_tokenizer = BertWordPieceTokenizer()
bert_wordpiece_tokenizer.train(
    files=[corpus_path], vocab_size=vocab_size)
bert_wordpiece_tokenizer.save_model(
    directory='./huggingface_output', prefix='BertWordPieceTokenizer-covid')

['./huggingface_output/BertWordPieceTokenizer-covid-vocab.txt']

In [128]:
sent_ko = '신종 코로나바이러스 감염증(코로나19) 사태가 심각합니다'
tokenizers = [bert_wordpiece_tokenizer,
              sentencepiece_bpe_tokenizer,
              char_bpe_tokenizer,
              byte_level_bpe_tokenizer]

for tokenizer in tokenizers:
    encode_single = tokenizer.encode(sent_ko)
    print(f'\n{tokenizer.__class__.__name__}')
    print(f'tokens = {encode_single.tokens}')


BertWordPieceTokenizer
tokens = ['신종', '코로나바이러스', '감염증', '(', '코로나19', ')', '사태', '##가', '심', '##각', '##합니다']

SentencePieceBPETokenizer
tokens = ['▁신종', '▁코로나바이러스', '▁감염증(코로나19)', '▁사태', '가', '▁심', '각', '합', '니다']

CharBPETokenizer
tokens = ['신종</w>', '코로나바이러스</w>', '감염증</w>', '(</w>', '코로나19</w>', ')</w>', '사태', '가</w>', '심', '각', '합니다</w>']

ByteLevelBPETokenizer
tokens = ['ìĭłì¢ħ', 'Ġì½Ķë¡ľëĤĺë°ĶìĿ´ëŁ¬ìĬ¤', 'Ġê°ĲìĹ¼ì¦Ŀ', '(', 'ì½Ķë¡ľëĤĺ', '19', ')', 'ĠìĤ¬íĥľ', 'ê°Ģ', 'Ġìĭ¬', 'ê°ģ', 'íķ©ëĭĪëĭ¤']


## 학습한 토크나이저를 transformers 에서 이용

- 위에서 train 한 bert_wordpiece_tokenizer 의 encode() method 와 저장된 vocab file을 load한 tokenizer의 tokenization이 동일함을 확인

In [132]:
from transformers import BertTokenizer, GPT2Tokenizer

transformers_bert_tokenizer = BertTokenizer(
    vocab_file = './huggingface_output/BertWordPieceTokenizer-covid-vocab.txt'
)
print(f'tokenizers  : {bert_wordpiece_tokenizer.encode(sent_ko).tokens}')
print(f'transformers: {transformers_bert_tokenizer.tokenize(sent_ko)}')

tokenizers  : ['신종', '코로나바이러스', '감염증', '(', '코로나19', ')', '사태', '##가', '심', '##각', '##합니다']
transformers: ['신종', '코로나바이러스', '감염증', '(', '코로나19', ')', '사태', '##가', '심', '##각', '##합니다']


화면에서 한글로 보이지만 실제로는 자/모 분해가 된 글자입니다

In [133]:
for token in bert_wordpiece_tokenizer.encode(sent_ko).tokens[:3]:
    print(f'len({token}) = {len(token)}')

len(신종) = 6
len(코로나바이러스) = 14
len(감염증) = 9


- unicodedata.normalize  


- ‘NFKD’ 는 한글의 자/모를 분해, ‘NFKC’ 는 자/모를 한글로 조합합니다

In [136]:
from unicodedata import normalize

print(normalize('NFKD', '가감'))  # 출력 시 글자를 재조합해서 보여줌
print(len(normalize('NFKD', '가감')))

print(normalize('NFKC', normalize('NFKD', '가감')))
print(len(normalize('NFKC', normalize('NFKD', '가감'))))

가감
5
가감
2


매번 NFKC 로 후처리를 반복해야 한다면 간단한 함수를 만들어 두는 것이 편리

In [137]:
from unicodedata import normalize

def compose(tokens):
    return [normalize('NFKC', token) for token in tokens]

print(f'tokenizers  : {compose(bert_wordpiece_tokenizer.encode(sent_ko).tokens)}')
print(f'transformers: {compose(transformers_bert_tokenizer.tokenize(sent_ko))}')

tokenizers  : ['신종', '코로나바이러스', '감염증', '(', '코로나19', ')', '사태', '##가', '심', '##각', '##합니다']
transformers: ['신종', '코로나바이러스', '감염증', '(', '코로나19', ')', '사태', '##가', '심', '##각', '##합니다']


In [139]:
transformers_gpt2_tokenizer = GPT2Tokenizer(
    vocab_file = './huggingface_output/ByteLevelBPETokenizer-covid-vocab.json',
    merges_file = './huggingface_output/ByteLevelBPETokenizer-covid-merges.txt'
)
print(f'tokenizers  : {byte_level_bpe_tokenizer.encode(sent_ko).tokens}')
print(f'transformers: {transformers_gpt2_tokenizer.tokenize(sent_ko)}')

tokenizers  : ['ìĭłì¢ħ', 'Ġì½Ķë¡ľëĤĺë°ĶìĿ´ëŁ¬ìĬ¤', 'Ġê°ĲìĹ¼ì¦Ŀ', '(', 'ì½Ķë¡ľëĤĺ', '19', ')', 'ĠìĤ¬íĥľ', 'ê°Ģ', 'Ġìĭ¬', 'ê°ģ', 'íķ©ëĭĪëĭ¤']
transformers: ['ìĭłì¢ħ', 'Ġì½Ķë¡ľëĤĺë°ĶìĿ´ëŁ¬ìĬ¤', 'Ġê°ĲìĹ¼ì¦Ŀ', '(', 'ì½Ķë¡ľëĤĺ', '19', ')', 'ĠìĤ¬íĥľ', 'ê°Ģ', 'Ġìĭ¬', 'ê°ģ', 'íķ©ëĭĪëĭ¤']
